# 🏦 Banking Finance QLoRA Fine-Tuning
## Mistral-7B-Instruct on Banking & Finance QA Dataset

**Author:** Rakesh Madasani  
**Model:** `mistralai/Mistral-7B-Instruct-v0.3`  
**Dataset:** `RakeshMadasani/banking-finance-qa-dataset`  
**Hardware:** Google Colab T4 GPU  
**Output:** `RakeshMadasani/banking-finance-mistral-qlora`

---

### 📋 Pipeline Overview
```
Step 1 → Install dependencies
Step 2 → Authenticate Hugging Face
Step 3 → Load & inspect dataset
Step 4 → Load Mistral-7B in 4-bit (QLoRA)
Step 5 → Configure LoRA adapters
Step 6 → Format prompts (Alpaca → Mistral Instruct)
Step 7 → Train with SFTTrainer
Step 8 → Evaluate (loss, sample outputs)
Step 9 → Push adapter to Hugging Face Hub
Step 10 → Test inference
```

### ⚡ Runtime Setup
> **Runtime → Change runtime type → T4 GPU** (free tier)  
> Estimated training time: ~90-120 minutes on T4

---

## Step 0: Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f"\n✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ VRAM: {total:.1f} GB")
else:
    print("❌ No GPU found. Go to Runtime → Change runtime type → T4 GPU")

## Step 1: Install Dependencies

In [ ]:
# Install all required packages
# This takes ~3-5 minutes on first run
!pip install -q \
    transformers==4.44.0 \
    datasets==2.20.0 \
    peft==0.12.0 \
    trl==0.9.6 \
    bitsandbytes==0.43.3 \
    accelerate==0.33.0 \
    scipy \
    sentencepiece \
    protobuf \
    huggingface_hub

print("✅ All packages installed!")

In [ ]:
# Verify versions
import transformers, datasets, peft, trl, bitsandbytes, accelerate
print(f"transformers : {transformers.__version__}")
print(f"datasets     : {datasets.__version__}")
print(f"peft         : {peft.__version__}")
print(f"trl          : {trl.__version__}")
print(f"bitsandbytes : {bitsandbytes.__version__}")
print(f"accelerate   : {accelerate.__version__}")

## Step 2: Authenticate Hugging Face

> Get your token from: https://huggingface.co/settings/tokens  
> Create a **write** token (needed to push the fine-tuned model)

In [ ]:
from huggingface_hub import login, HfApi
from google.colab import userdata

# Option 1: Use Colab Secrets (recommended)
# Go to the key icon (🔑) in the left sidebar → Add secret → Name: HF_TOKEN
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("✅ Logged in via Colab Secrets")
except Exception:
    # Option 2: Manual input
    print("Colab secret not found. Enter your HF token manually:")
    login()  # Will prompt for token input

# Verify login
api = HfApi()
user = api.whoami()
print(f"✅ Logged in as: {user['name']}")

## Step 3: Configuration — All Settings in One Place

Use `MAX_SAMPLES = 500` for the first run on Colab T4, then switch to `None` for the full run.


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║           PROJECT CONFIGURATION — EDIT HERE             ║
# ╚══════════════════════════════════════════════════════════╝

# ── Model ────────────────────────────────────────────────────
BASE_MODEL      = "mistralai/Mistral-7B-Instruct-v0.3"
NEW_MODEL_NAME  = "banking-finance-mistral-qlora"          # repo name on HF
HF_USERNAME     = "RakeshMadasani"                          # your HF username
HF_REPO_ID      = f"{HF_USERNAME}/{NEW_MODEL_NAME}"

# ── Dataset ───────────────────────────────────────────────────
DATASET_NAME    = "RakeshMadasani/banking-finance-qa-dataset"
TRAIN_SPLIT     = "train"
VAL_SPLIT       = "validation"
MAX_SAMPLES     = 500   # first test run; set None for full training later

# ── QLoRA / bitsandbytes ──────────────────────────────────────
LOAD_IN_4BIT    = True
BNB_4BIT_DTYPE  = "nf4"           # nf4 or fp4
USE_NESTED_QUANT = True           # double quantization saves VRAM
COMPUTE_DTYPE   = "float16"       # safer default for Colab T4

# ── LoRA Hyperparameters ──────────────────────────────────────
LORA_R          = 16              # rank (16-64 typical; lower = fewer params)
LORA_ALPHA      = 32              # scaling factor (usually 2x rank)
LORA_DROPOUT    = 0.05            # dropout on LoRA layers
LORA_BIAS       = "none"          # none, all, or lora_only
TARGET_MODULES  = [               # Mistral attention + MLP layers
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj"
]

# ── Training Hyperparameters ──────────────────────────────────
OUTPUT_DIR      = "./banking-mistral-qlora"
NUM_EPOCHS      = 2               # first test run; switch to 3 for full training later
PER_DEVICE_BATCH = 1              # safer starting point for Colab T4
GRAD_ACCUM      = 8               # effective batch = 2 × 8 = 16
LEARNING_RATE   = 2e-4            # standard QLoRA learning rate
LR_SCHEDULER    = "cosine"        # cosine decay
WARMUP_RATIO    = 0.05            # 5% warmup steps
MAX_SEQ_LEN     = 512             # max token length per sample
WEIGHT_DECAY    = 0.001
OPTIMIZER       = "paged_adamw_8bit"  # memory-efficient optimizer
SAVE_STEPS      = 100
LOGGING_STEPS   = 25
EVAL_STEPS      = 100
FP16            = True
BF16            = False           # use FP16 on Colab T4
GRADIENT_CKPT   = True            # saves VRAM at cost of slight speed
PACKING         = False           # False = one sample per sequence

print("✅ Configuration loaded!")
print(f"   Base model   : {BASE_MODEL}")
print(f"   Dataset      : {DATASET_NAME}")
print(f"   Output repo  : {HF_REPO_ID}")
print(f"   LoRA rank    : {LORA_R}")
print(f"   Epochs       : {NUM_EPOCHS}")
print(f"   Batch size   : {PER_DEVICE_BATCH} × {GRAD_ACCUM} = {PER_DEVICE_BATCH*GRAD_ACCUM} effective")

## Step 4: Load and Inspect Dataset

In [ ]:
from datasets import load_dataset
import pandas as pd

print(f"Loading dataset: {DATASET_NAME}")
dataset = load_dataset(DATASET_NAME)
print(f"\n✅ Dataset loaded!")
print(dataset)

train_data = dataset[TRAIN_SPLIT]
val_data   = dataset[VAL_SPLIT]

# Optional: limit samples for quick test
if MAX_SAMPLES is not None:
    train_data = train_data.select(range(min(MAX_SAMPLES, len(train_data))))
    val_data   = val_data.select(range(min(MAX_SAMPLES // 10, len(val_data))))
    print(f"\n⚠️ Limited to {MAX_SAMPLES} samples for testing")

print(f"\nTrain samples : {len(train_data)}")
print(f"Val samples   : {len(val_data)}")

# Show sample
print("\n" + "="*60)
print("SAMPLE 1:")
print("="*60)
sample = train_data[0]
print(f"Instruction: {sample['instruction'][:200]}")
print(f"Input      : {sample.get('input', '')}")
print(f"Output     : {sample['output'][:300]}")

In [ ]:
# Analyze dataset statistics
import numpy as np

q_lens = [len(x['instruction'].split()) for x in train_data]
a_lens = [len(x['output'].split()) for x in train_data]

print("📊 Dataset Statistics (Train)")
print(f"Question word length — min: {min(q_lens)}, avg: {np.mean(q_lens):.0f}, max: {max(q_lens)}")
print(f"Answer word length   — min: {min(a_lens)}, avg: {np.mean(a_lens):.0f}, max: {max(a_lens)}")

# Show class distribution if 'type' column exists
if 'type' in train_data.column_names:
    from collections import Counter
    types = Counter(train_data['type'])
    print("\n📋 Question Types:")
    for k, v in types.most_common():
        print(f"  {k:<15}: {v:>5} ({100*v/len(train_data):.1f}%)")

print(f"\n✅ Dataset looks good for fine-tuning!")

## Step 5: Format Prompts (Alpaca → Mistral Instruct Format)

In [ ]:
# Prompt template helper
SYSTEM_PROMPT = (
    "You are a banking and finance expert assistant with deep knowledge of "
    "Indian banking regulations (RBI, PMLA, KYC, Basel norms) and U.S. banking regulations "
    "(BSA, FinCEN, FDIC, OCC, CFPB, Dodd-Frank). Provide accurate, detailed, and professional answers."
)

def format_prompt(sample):
    instruction = sample["instruction"].strip()
    inp = sample.get("input", "").strip()
    output = sample["output"].strip()

    user_content = instruction if not inp else f"{instruction}\n\nContext: {inp}"

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": output},
    ]

    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": formatted}

print("Prompt formatter ready. Run formatting after the tokenizer loads.")


## Step 6: Load Mistral-7B in 4-bit Quantization (QLoRA)

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

compute_dtype = getattr(torch, COMPUTE_DTYPE)

# ── 4-bit Quantization Configuration ─────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=LOAD_IN_4BIT,
    bnb_4bit_quant_type=BNB_4BIT_DTYPE,           # nf4: best for LLMs
    bnb_4bit_compute_dtype=compute_dtype,           # bfloat16 for T4
    bnb_4bit_use_double_quant=USE_NESTED_QUANT,    # saves ~0.5 GB VRAM
)

print(f"Loading {BASE_MODEL}...")
print("(This will download ~14GB on first run — takes 5-10 minutes)")

# ── Load Model ────────────────────────────────────────────────
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",           # automatically place on GPU
    trust_remote_code=True,
    torch_dtype=compute_dtype,
)
model.config.use_cache = False              # disable KV cache for training
model.config.pretraining_tp = 1            # tensor parallelism = 1 (single GPU)

# ── Load Tokenizer ────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token  # set pad token
tokenizer.padding_side = "right"           # right padding for causal LM

# ── Check VRAM usage ─────────────────────────────────────────
allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved() / 1e9
print(f"\n✅ Model loaded!")
print(f"   VRAM allocated : {allocated:.1f} GB")
print(f"   VRAM reserved  : {reserved:.1f} GB")
print(f"   Model dtype    : {model.dtype}")
print(f"   Parameters     : {sum(p.numel() for p in model.parameters()):,}")
# Format dataset after tokenizer initialization
train_formatted = train_data.map(format_prompt, remove_columns=train_data.column_names)
val_formatted   = val_data.map(format_prompt, remove_columns=val_data.column_names)

print("Prompt formatting complete.")
print("\nSample formatted prompt (first 500 chars):")
print("-" * 60)
print(train_formatted[0]["text"][:500])
print("-" * 60)
print(f"\nAvg text length: {np.mean([len(x['text'].split()) for x in train_formatted]):.0f} words")


## Step 7: Configure LoRA Adapters

In [ ]:
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType
)

# ── Prepare model for k-bit training ─────────────────────────
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=GRADIENT_CKPT
)

# ── LoRA Configuration ────────────────────────────────────────
lora_config = LoraConfig(
    r=LORA_R,                           # rank — controls number of trainable params
    lora_alpha=LORA_ALPHA,              # scaling (alpha/r = effective learning rate scale)
    target_modules=TARGET_MODULES,      # which layers to apply LoRA to
    lora_dropout=LORA_DROPOUT,          # regularization
    bias=LORA_BIAS,                     # bias terms
    task_type=TaskType.CAUSAL_LM,       # causal language modeling
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# ── Count trainable parameters ────────────────────────────────
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
pct       = 100 * trainable / total

print("✅ LoRA adapters configured!")
print(f"   Trainable params : {trainable:,} ({pct:.2f}% of total)")
print(f"   Total params     : {total:,}")
print(f"   LoRA rank (r)    : {LORA_R}")
print(f"   LoRA alpha       : {LORA_ALPHA}")
print(f"   Target modules   : {TARGET_MODULES}")

# This is the magic of QLoRA:
# Only ~0.5-1% of parameters are actually trained!
model.print_trainable_parameters()

## Step 8: Configure Training Arguments

In [ ]:
from transformers import TrainingArguments
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    # ── Output ─────────────────────────────────────────
    output_dir=OUTPUT_DIR,
    
    # ── Training Duration ───────────────────────────────
    num_train_epochs=NUM_EPOCHS,
    
    # ── Batch & Gradient ────────────────────────────────
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=GRADIENT_CKPT,
    
    # ── Learning Rate ───────────────────────────────────
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_ratio=WARMUP_RATIO,
    
    # ── Optimizer ──────────────────────────────────────
    optim=OPTIMIZER,              # paged_adamw_8bit saves VRAM
    
    # ── Precision ──────────────────────────────────────
    fp16=FP16,
    bf16=BF16,
    
    # ── Logging & Saving ───────────────────────────────
    logging_steps=LOGGING_STEPS,
    logging_dir=f"{OUTPUT_DIR}/logs",
    save_steps=SAVE_STEPS,
    save_total_limit=2,           # keep only 2 checkpoints (saves disk)
    load_best_model_at_end=True,
    
    # ── Evaluation ─────────────────────────────────────
    evaluation_strategy="steps",
    eval_steps=EVAL_STEPS,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    # ── Misc ───────────────────────────────────────────
    group_by_length=True,         # group similar-length sequences (efficiency)
    report_to="none",             # disable wandb/tensorboard by default
    push_to_hub=False,            # we'll push manually in Step 10
    dataloader_pin_memory=False,  # reduces RAM usage
)

print("✅ Training arguments configured!")
total_steps = (len(train_formatted) // (PER_DEVICE_BATCH * GRAD_ACCUM)) * NUM_EPOCHS
print(f"   Effective batch size : {PER_DEVICE_BATCH * GRAD_ACCUM}")
print(f"   Estimated total steps: ~{total_steps}")
print(f"   Learning rate        : {LEARNING_RATE}")
print(f"   LR scheduler         : {LR_SCHEDULER}")

## Step 9: Initialize SFTTrainer and Train!

In [ ]:
from trl import SFTTrainer

# ── Initialize SFTTrainer ─────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_formatted,
    eval_dataset=val_formatted,
    args=training_args,
    dataset_text_field="text",          # column containing formatted prompts
    max_seq_length=MAX_SEQ_LEN,         # truncate/pad to 512 tokens
    packing=PACKING,                    # False: one sample per sequence
)

print("✅ SFTTrainer initialized!")
print(f"   Training samples : {len(train_formatted)}")
print(f"   Validation samples: {len(val_formatted)}")
print("\n🚀 Starting training...")
print("   Estimated time on T4: ~90-120 minutes for 3 epochs")
print("   Loss should decrease from ~2.0 to ~0.8-1.2")
print("-" * 60)

In [ ]:
# ── START TRAINING ────────────────────────────────────────────
import time
start_time = time.time()

train_result = trainer.train()

elapsed = time.time() - start_time
hours = elapsed // 3600
mins  = (elapsed % 3600) // 60

print(f"\n✅ Training complete!")
print(f"   Total time       : {int(hours)}h {int(mins)}m")
print(f"   Final train loss : {train_result.training_loss:.4f}")
print(f"   Total steps      : {train_result.global_step}")
print(f"   Samples/second   : {train_result.metrics.get('train_samples_per_second', 'N/A')}")

## Step 10: Evaluate Training Results

In [ ]:
import math

# ── Final Evaluation ──────────────────────────────────────────
print("Running final evaluation on validation set...")
eval_results = trainer.evaluate()

eval_loss = eval_results['eval_loss']
perplexity = math.exp(eval_loss)

print(f"\n📊 Evaluation Results:")
print(f"   Eval Loss   : {eval_loss:.4f}")
print(f"   Perplexity  : {perplexity:.2f}")
print(f"   Runtime     : {eval_results.get('eval_runtime', 0):.1f}s")

# ── Interpretation ────────────────────────────────────────────
print("\n📈 Model Quality Assessment:")
if eval_loss < 1.0:
    print("   ✅ Excellent — model learned banking domain very well")
elif eval_loss < 1.5:
    print("   ✅ Good — model learned banking domain well")
elif eval_loss < 2.0:
    print("   ⚠️  Fair — model learned but could use more training")
else:
    print("   ❌ Needs improvement — consider more epochs or higher LR")

In [ ]:
# ── Plot Training Loss Curve ──────────────────────────────────
import matplotlib.pyplot as plt

# Extract loss history from trainer logs
log_history = trainer.state.log_history

train_steps  = [x['step'] for x in log_history if 'loss' in x]
train_losses = [x['loss'] for x in log_history if 'loss' in x]
eval_steps   = [x['step'] for x in log_history if 'eval_loss' in x]
eval_losses  = [x['eval_loss'] for x in log_history if 'eval_loss' in x]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Banking Finance QLoRA — Training Curves', fontsize=14, fontweight='bold')

# Training loss
axes[0].plot(train_steps, train_losses, 'b-', linewidth=1.5, label='Train Loss')
if eval_losses:
    axes[0].plot(eval_steps, eval_losses, 'r-o', linewidth=2, markersize=5, label='Eval Loss')
axes[0].set_xlabel('Steps')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss vs Training Steps')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Learning rate schedule
lr_steps  = [x['step'] for x in log_history if 'learning_rate' in x]
lr_values = [x['learning_rate'] for x in log_history if 'learning_rate' in x]
if lr_steps:
    axes[1].plot(lr_steps, lr_values, 'g-', linewidth=2)
    axes[1].set_xlabel('Steps')
    axes[1].set_ylabel('Learning Rate')
    axes[1].set_title('Learning Rate Schedule (Cosine)')
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/training_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Training curves saved!")

## Step 11: Test Inference Before Saving

In [ ]:
from transformers import pipeline
import torch

# ── Set model to evaluation mode ─────────────────────────────
model.eval()

def generate_answer(question, max_new_tokens=256, temperature=0.1):
    """Generate answer for a banking question."""
    prompt = f"<s>[INST] {SYSTEM_PROMPT}\n\n{question} [/INST]"
    
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LEN
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )
    
    # Decode only the new tokens (not the prompt)
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# ── Test Questions ────────────────────────────────────────────
test_questions = [
    "What is the Repo Rate in India and what is its current value?",
    "What is a Suspicious Activity Report (SAR) and when must it be filed?",
    "What is the difference between CRR and SLR?",
    "What is the minimum Capital Adequacy Ratio required by RBI under Basel III?",
    "What are the three stages of money laundering?",
    "What is the FDIC deposit insurance limit in the United States?",
    "A bank has Gross NPAs of Rs. 5,000 crore and provisions of Rs. 4,000 crore. Calculate PCR.",
    "What is the difference between MCLR and EBLR?",
]

print("🧪 Testing fine-tuned model on banking questions...")
print("=" * 70)

for i, question in enumerate(test_questions, 1):
    print(f"\n[Q{i}] {question}")
    print("-" * 50)
    answer = generate_answer(question)
    print(f"[A{i}] {answer[:400]}")
    if len(answer) > 400:
        print("     [... truncated ...]")

print("\n" + "=" * 70)
print("✅ Inference test complete!")

## Step 12: Save Model and Push to Hugging Face Hub

In [ ]:
# ── Save LoRA adapter locally ─────────────────────────────────
print("Saving LoRA adapter locally...")
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Saved to {OUTPUT_DIR}/")

# Check saved files
import os
files = os.listdir(OUTPUT_DIR)
print(f"\nSaved files:")
for f in sorted(files):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1e6
    print(f"   {f:<40} {size:.1f} MB")

In [ ]:
# ── Create Model Card (README.md) ─────────────────────────────
model_card = f"""---
license: apache-2.0
base_model: {BASE_MODEL}
tags:
- banking
- finance
- india
- qlora
- lora
- mistral
- fine-tuned
- rbi
- aml
- kyc
- basel
language:
- en
datasets:
- {DATASET_NAME}
pipeline_tag: text-generation
---

# 🏦 Banking Finance Mistral-7B QLoRA

**Author:** [Rakesh Madasani](https://huggingface.co/{HF_USERNAME})  
**Base model:** [{BASE_MODEL}](https://huggingface.co/{BASE_MODEL})  
**Dataset:** [{DATASET_NAME}](https://huggingface.co/datasets/{DATASET_NAME})  
**Method:** QLoRA (Quantized Low-Rank Adaptation)

## 📋 Model Description

This is a fine-tuned version of Mistral-7B-Instruct for the **banking and finance domain**, 
specifically covering:

- 🇮🇳 **Indian Banking Regulations** — RBI, PMLA, KYC, Credit Risk, Basel Norms, NPAs, Financial Ratios
- 🇺🇸 **U.S. Banking Regulations** — BSA, FinCEN, FDIC, OCC, CFPB, Dodd-Frank, OFAC, SAR/CTR
- 📊 **Financial Analysis** — NIM, ROA, ROE, NPA ratios, PCR, CASA, Basel III capital calculations
- 🔄 **AML/KYC** — Money laundering stages, suspicious activity detection, EDD, PEPs
- 📐 **Calculations** — EMI, CAR, NIM, PCR, LCR, NSFR, Expected Loss

## ⚙️ Training Details

| Parameter | Value |
|-----------|-------|
| Base Model | Mistral-7B-Instruct-v0.3 |
| Fine-tuning Method | QLoRA (4-bit NF4) |
| Training Dataset | {DATASET_NAME} |
| Training Samples | ~3,000 |
| LoRA Rank (r) | {LORA_R} |
| LoRA Alpha | {LORA_ALPHA} |
| Epochs | {NUM_EPOCHS} |
| Learning Rate | {LEARNING_RATE} |
| Max Seq Length | {MAX_SEQ_LEN} |
| Hardware | Google Colab T4 GPU |
| Trainer | TRL SFTTrainer |

## 🚀 Usage

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    "{BASE_MODEL}",
    torch_dtype=torch.float16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("{BASE_MODEL}")

# Load fine-tuned LoRA adapter
model = PeftModel.from_pretrained(base_model, "{HF_REPO_ID}")
model.eval()

# Generate answer
question = "What is the Repo Rate in India?"
prompt = f"<s>[INST] {{question}} [/INST]"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=256)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))
```

## 📊 Example Q&A

**Q:** What is the Capital Adequacy Ratio (CAR) requirement in India under Basel III?  
**A:** Under Basel III, the minimum Capital Adequacy Ratio (CAR) in India is 9%, which is higher than the global Basel III minimum of 8%. RBI's stricter requirement provides an additional buffer for Indian banks. CAR = (Tier 1 Capital + Tier 2 Capital) / Risk Weighted Assets × 100%.

**Q:** What is a Suspicious Activity Report (SAR) in the U.S.?  
**A:** A SAR must be filed with FinCEN when a bank suspects a transaction of $5,000 or more involves illegal activity. It must be filed within 30 days of detection and is strictly confidential — banks cannot inform customers that a SAR was filed.

## 📁 Project

This model is **Project 3** of a 3-part Banking AI portfolio:
1. 🔍 [Banking RAG System](https://huggingface.co/spaces/{HF_USERNAME}/banking-finance-rag)
2. 📊 [Banking QA Dataset](https://huggingface.co/datasets/{DATASET_NAME})
3. 🤖 **This model** — Banking Finance QLoRA Fine-Tune

## ⚠️ Limitations

- This model is for educational and portfolio demonstration purposes
- Not intended for actual regulatory compliance or financial decisions
- Regulatory information may have changed since training data was created
- Always verify with official regulatory sources (RBI, FinCEN, FDIC, etc.)
"""

with open(f"{OUTPUT_DIR}/README.md", 'w') as f:
    f.write(model_card)

print("✅ Model card (README.md) created!")

In [ ]:
# ── Push LoRA adapter to Hugging Face Hub ─────────────────────
from huggingface_hub import HfApi, create_repo

print(f"Pushing model to: https://huggingface.co/{HF_REPO_ID}")

# Create repo if it doesn't exist
try:
    create_repo(HF_REPO_ID, repo_type="model", exist_ok=True)
    print(f"✅ Repository ready: {HF_REPO_ID}")
except Exception as e:
    print(f"Note: {e}")

# Push adapter weights and tokenizer
trainer.model.push_to_hub(
    HF_REPO_ID,
    commit_message="Add banking finance QLoRA adapter (Mistral-7B)"
)
tokenizer.push_to_hub(
    HF_REPO_ID,
    commit_message="Add tokenizer"
)

# Upload model card
api = HfApi()
api.upload_file(
    path_or_fileobj=f"{OUTPUT_DIR}/README.md",
    path_in_repo="README.md",
    repo_id=HF_REPO_ID,
    repo_type="model",
    commit_message="Add model card"
)

# Upload training curves
try:
    api.upload_file(
        path_or_fileobj=f"{OUTPUT_DIR}/training_curves.png",
        path_in_repo="training_curves.png",
        repo_id=HF_REPO_ID,
        repo_type="model",
    )
except Exception:
    pass

print(f"\n🎉 Model successfully pushed to Hugging Face!")
print(f"   View at: https://huggingface.co/{HF_REPO_ID}")

## Step 13: Merge Adapter with Base Model (Optional — for full model upload)

In [ ]:
# ── OPTIONAL: Merge LoRA weights into base model ──────────────
# This creates a standalone model (no need for PEFT library to run)
# Requires ~2x VRAM — only run if you have enough memory
# Skip this cell if you just want the LoRA adapter (smaller, recommended)

MERGE_AND_UPLOAD = False  # Set to True to merge and upload full model

if MERGE_AND_UPLOAD:
    from peft import PeftModel
    from transformers import AutoModelForCausalLM
    import torch
    
    print("Loading base model for merging (requires ~14GB VRAM)...")
    
    # Load base model in float16 for merging
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    
    # Load LoRA adapter
    peft_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
    
    # Merge weights
    merged_model = peft_model.merge_and_unload()
    
    MERGED_DIR = "./banking-mistral-merged"
    merged_model.save_pretrained(MERGED_DIR)
    tokenizer.save_pretrained(MERGED_DIR)
    
    print(f"✅ Merged model saved to {MERGED_DIR}/")
    print("   You can upload this to a separate HF repo as a full model")
else:
    print("Skipping merge (MERGE_AND_UPLOAD = False)")
    print("The LoRA adapter is sufficient for inference with PEFT library.")

## Step 14: Final Summary and Next Steps

In [ ]:
print("=" * 70)
print("🏆 BANKING FINANCE QLoRA FINE-TUNING COMPLETE!")
print("=" * 70)

print(f"""
📊 TRAINING SUMMARY
{'─'*40}
Base Model     : {BASE_MODEL}
Dataset        : {DATASET_NAME}
Training pairs : {len(train_formatted)}
LoRA rank (r)  : {LORA_R}
LoRA alpha     : {LORA_ALPHA}
Epochs         : {NUM_EPOCHS}
Final eval loss: {eval_results.get('eval_loss', 'N/A'):.4f}
Perplexity     : {perplexity:.2f}

📦 OUTPUT
{'─'*40}
HF Model Repo  : https://huggingface.co/{HF_REPO_ID}
Local backup   : {OUTPUT_DIR}/

✅ PORTFOLIO STATUS
{'─'*40}
Project 1 ✅   : Banking RAG System (LIVE)
               : https://huggingface.co/spaces/{HF_USERNAME}/banking-finance-rag
Project 2 ✅   : Banking QA Dataset (LIVE — 3,002 pairs)
               : https://huggingface.co/datasets/{DATASET_NAME}
Project 3 ✅   : QLoRA Fine-Tune (LIVE — Mistral-7B)
               : https://huggingface.co/{HF_REPO_ID}

📝 RESUME BULLETS
{'─'*40}
• Fine-tuned Mistral-7B on 3,002-pair custom banking domain dataset
  using QLoRA (4-bit NF4 quantization), achieving {perplexity:.1f}x perplexity
  on held-out banking Q&A validation set; deployed on Hugging Face Hub

• Built end-to-end LLM fine-tuning pipeline: dataset curation →
  QLoRA training (LoRA r=16) → evaluation → HF Hub deployment
  covering Indian/US banking regulations (RBI, BSA, FDIC, Basel III)
""")

print("=" * 70)
print("All 3 projects complete! 🎉")
print("=" * 70)

---

## 🔧 Troubleshooting

| Issue | Solution |
|-------|----------|
| CUDA out of memory | Reduce `PER_DEVICE_BATCH` to 1 or `MAX_SEQ_LEN` to 384 |
| Training very slow | Ensure T4 GPU is selected: Runtime → Change runtime type |
| Loss not decreasing | Try increasing `LEARNING_RATE` to 3e-4 or more epochs |
| HF push fails | Check token has **write** permissions at hf.co/settings/tokens |
| `bitsandbytes` error | Restart runtime and re-run from Step 1 |
| Colab disconnects | Enable "Stay connected" Chrome extension; save checkpoints |
| Out of disk space | Run `!df -h` to check; reduce `save_total_limit` to 1 |

## 📚 Key Resources

- [QLoRA Paper](https://arxiv.org/abs/2305.14314)
- [Mistral-7B-Instruct](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3)
- [PEFT Documentation](https://huggingface.co/docs/peft)
- [TRL SFTTrainer](https://huggingface.co/docs/trl/sft_trainer)
- [BitsAndBytes](https://github.com/TimDettmers/bitsandbytes)